In [1]:
!ls ../data/wiki*.jsonl

../data/wikipedia_synthetic1.jsonl  ../data/wikipedia_synthetic4.jsonl
../data/wikipedia_synthetic2.jsonl  ../data/wikipedia_synthetic5.jsonl
../data/wikipedia_synthetic3.jsonl  ../data/wikipedia_synthetic6.jsonl


In [2]:
import json
import pandas as pd
from glob import glob

PATH = "../data/wiki*.jsonl"
files = glob(PATH)
# files = [f for f in files if "8" in f or "9" in f ]

def load_json(file):
    def get_data(raw):
        line = json.loads(raw)
        return {
            "text": line["text"],
            "labels": line.get("labels"),
            "not_labels": line.get("not_labels")
        }
    with open(file, "r") as f:
        data = [get_data(line) for line in f]
    return data

files_data = [load_json(file) for file in files]
df = pd.DataFrame([i for file_data in files_data for i in file_data])

In [3]:
df.sample(5)

,text,labels,not_labels
290,Step onto the pitch at Lord's where a young fa...,"[breakthrough moment, victory celebration, sen...","[injury interruption, retirement closure, nost..."
8216,There's a contradiction in your records that n...,"[pointing_out_contradiction, questioning_accur...","[requesting_correction, expressing_urgency, de..."
18206,"Construction began in the twelfth century, but...","[chronological_sequencing, artistic_craftsmans...","[spatial_description, stylistic_synthesis, tra..."
3991,I purchased tickets for the North Wales Intern...,"[dissatisfaction_with_artistic_programming, cr...","[grievance_about_physical_accessibility, frust..."
16999,Some patients experience a milk-colored discha...,"[reassurance_element, complication_alert, caus...","[directive_instruction, timeline_expectation, ..."


In [4]:
def merge_group(group):
    merged_labels = set().union(*group["labels"])
    merged_not_labels = set().union(*group["not_labels"])
    merged_not_labels -= merged_labels  # remove any intersection
    return pd.Series({
        "labels": sorted(merged_labels),
        "not_labels": sorted(merged_not_labels),
    })

df = (
    df.groupby("text", sort=False)
    .apply(merge_group, include_groups=False)
    .reset_index()
)
print(f"{len(df)} unique texts after merging")
df.sample(5)


19249 unique texts after merging


,text,labels,not_labels
18303,Appointments to academic positions shall be ma...,"[appointment_authority, eligibility_criteria, ...","[charitable_purpose, collaborative_obligation,..."
5415,Original pharmaceutical documentation for the ...,"[archival_documentation, institutional_medical...","[commercial_promotional_material, medical_prac..."
2635,Two rivers meet and carry me away\nVipava sing...,"[borderland_identity, confluence_metaphor, riv...","[sacred_space, twin_connection, underground_de..."
18658,"Everyone called her a journeyman, just passing...","[crossing_point, reclassification_claim, stati...","[achievement_celebration, momentum_buildup, su..."
8210,The regional containment strategy implemented ...,"[market_containment, operational_constraint, p...","[comparative_analysis, forward_looking_stateme..."


In [5]:
import random
from collections import Counter, defaultdict

random.seed(42)
test_ratio = 0.1

# ── 1. Label frequency overview ───────────────────────────────────────────────
label_counts = Counter(lab for labs in df["labels"] for lab in labs)
not_label_counts = Counter(lab for labs in df["not_labels"] for lab in labs)

print(f"Unique positive labels : {len(label_counts)}")
print(f"Unique negative labels : {len(not_label_counts)}")
print(f"\nTop-10 positive labels:")
for lab, cnt in label_counts.most_common(10):
    print(f"  {lab:50s}  {cnt}")

# ── 2. Select held-out (test) labels ──────────────────────────────────────────
# Build label → row-index mapping
label_to_rows = defaultdict(set)
for i, labs in enumerate(df["labels"]):
    for lab in labs:
        label_to_rows[lab].add(i)

# Shuffle to avoid systematic bias, then greedily cover rows until target
all_labels = list(label_counts.keys())
random.shuffle(all_labels)

target_test_n = int(test_ratio * len(df))
test_labels = set()
test_row_indices = set()

for label in all_labels:
    if len(test_row_indices) >= target_test_n:
        break
    new_rows = label_to_rows[label] - test_row_indices
    if new_rows:
        test_labels.add(label)
        test_row_indices |= new_rows

train_labels = set(label_counts.keys()) - test_labels

print(f"\nLabel vocabulary split:")
print(f"  Train labels : {len(train_labels)}")
print(f"  Test labels  : {len(test_labels)}")

# ── 3. Assign rows ─────────────────────────────────────────────────────────────
# A row goes to test if ANY positive label is a held-out test label
is_test = df["labels"].apply(lambda labs: bool(set(labs) & test_labels))

df_train = df[~is_test].reset_index(drop=True)
df_test  = df[is_test].reset_index(drop=True)

print(f"\nRow split:")
print(f"  Train : {len(df_train):6d}  ({len(df_train) / len(df):.1%})")
print(f"  Test  : {len(df_test):6d}  ({len(df_test)  / len(df):.1%})")

# ── 4. Sanity check: zero label leakage ───────────────────────────────────────
train_positive_labels = set(lab for labs in df_train["labels"] for lab in labs)
leakage = test_labels & train_positive_labels
print(f"\nLabel leakage into train positives (must be 0): {len(leakage)}")


Unique positive labels : 42376
Unique negative labels : 34264

Top-10 positive labels:
  comparative_analysis                                126
  historical_documentation                            103
  heritage_preservation                               97
  legacy_preservation                                 82
  historical_continuity                               80
  institutional_critique                              79
  historical_context                                  78
  temporal_progression                                70
  institutional_affiliation                           69
  institutional_memory                                67

Label vocabulary split:
  Train labels : 41279
  Test labels  : 1097

Row split:
  Train :  17324  (90.0%)
  Test  :   1925  (10.0%)

Label leakage into train positives (must be 0): 0


In [6]:
import datasets

train_ds = datasets.Dataset.from_pandas(df_train)
test_ds = datasets.Dataset.from_pandas(df_test)

dataset = datasets.DatasetDict({
    "train": train_ds,
    "test": test_ds
})

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'not_labels'],
        num_rows: 17324
    })
    test: Dataset({
        features: ['text', 'labels', 'not_labels'],
        num_rows: 1925
    })
})

In [7]:
dataset.push_to_hub("alexneakameni/ZSHOT-HARDSET-v2", commit_description="Upload of ZSHOT-HARDSET-v2 with train/test split based on held-out labels.")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-v2/commit/1d8aedc4d03b6af64f0df48a85b8c3ac80f055a3', commit_message='Upload dataset', commit_description='Upload of ZSHOT-HARDSET-v2 with train/test split based on held-out labels.', oid='1d8aedc4d03b6af64f0df48a85b8c3ac80f055a3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alexneakameni/ZSHOT-HARDSET-v2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alexneakameni/ZSHOT-HARDSET-v2'), pr_revision=None, pr_num=None)